## Idea

This notebook trains the XGBoost model using the selected feature configuration
(no_lac_features) and compares three validation strategies:
- SGKF by day,
- rolling validation,
- standard SKF.

---

### Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import xgboost as xgb

import mlflow
import mlflow.xgboost

from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedKFold
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    classification_report
)

### Paths & MLFlow

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
DATA_PATH = PROJECT_ROOT / "data"

TRAIN_PATH = DATA_PATH / "dvc" / "train_v5.parquet"

mlflow.set_tracking_uri(
    f"file:{PROJECT_ROOT / 'mlruns'}"
)

mlflow.set_experiment(
    "simbox_fraud_detection_v5"
)

c:\Users\ali.farhat\AppData\Local\miniconda3\envs\simbox\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/08/08 00:40:42 INFO mlflow.tracking.fluent: Experiment with name 'simbox_fraud_detection_v5' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:c:\\Users\\ali.farhat\\Documents\\simbox-project\\mlruns/479133145194050450', creation_time=1786138842469, experiment_id='479133145194050450', last_update_time=1786138842469, lifecycle_stage='active', name='simbox_fraud_detection_v5', tags={}, trace_location=None, workspace='default'>

### Load dataset

In [3]:
df = pd.read_parquet(TRAIN_PATH)

df["profile_date"] = pd.to_datetime(
    df["profile_date"]
)

print(df.shape)

display(df.head())

(314610, 26)


,msisdn,label,nb_out_calls,nb_in_calls,nb_called_numbers,avg_out_duration,avg_in_duration,ratio_in_vs_out,ratio_distinct_vs_out,ratio_in_duration_vs_out_duration,...,ratio_afternoon_vs_out,ratio_night_vs_out,nb_flagged_calls,ratio_flagged_vs_out,nb_distinct_lacs,ratio_distinct_lacs_vs_out,nb_distinct_cell_ids,ratio_distinct_cell_ids_vs_out,avg_delay_per_day,profile_date
0,22362444330,0,0,5,0,0.000000,136.200000,999999.000000,0.000000,0.000000,...,999999.000000,999999.000000,5,999999.000000,1,999999.000000,2,999999.000000,0.000000,2022-11-01
1,2250152130667,0,3,0,3,39.000000,0.000000,0.000000,1.000000,0.000000,...,0.000000,1.000000,3,1.000000,2,0.666667,2,0.666667,763.500000,2022-11-01
2,2250152127834,1,9,3,8,81.777778,421.666667,0.333333,0.888889,5.156250,...,1.000000,0.111111,12,1.333333,1,0.111111,7,0.777778,3764.875000,2022-11-01
3,2250152127796,0,14,21,8,69.857143,109.095238,1.500000,0.571429,1.561691,...,0.357143,1.000000,20,1.428571,1,0.071429,7,0.500000,3721.384615,2022-11-01
4,2250152127414,0,0,1,0,0.000000,150.000000,999999.000000,0.000000,0.000000,...,0.000000,0.000000,1,999999.000000,1,999999.000000,1,999999.000000,0.000000,2022-11-01


### Prepare features

In [4]:
DROP_COLUMNS = [
    "msisdn",
    "profile_date",
    "label",

    # LAC ablation
    "nb_distinct_lacs",
    "ratio_distinct_lacs_vs_out"
]


features = [
    c for c in df.columns
    if c not in DROP_COLUMNS
]


print(f"Features used: {len(features)}")
print(features)

Features used: 21
['nb_out_calls', 'nb_in_calls', 'nb_called_numbers', 'avg_out_duration', 'avg_in_duration', 'ratio_in_vs_out', 'ratio_distinct_vs_out', 'ratio_in_duration_vs_out_duration', 'dawn_count', 'day_count', 'afternoon_count', 'night_count', 'ratio_dawn_vs_out', 'ratio_day_vs_out', 'ratio_afternoon_vs_out', 'ratio_night_vs_out', 'nb_flagged_calls', 'ratio_flagged_vs_out', 'nb_distinct_cell_ids', 'ratio_distinct_cell_ids_vs_out', 'avg_delay_per_day']


### XGBoost parameters

In [5]:
params = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "max_depth": 6,
    "scale_pos_weight": (
        (df.label == 0).sum()
        /
        (df.label == 1).sum()
    ),
    "seed": 42,
    "n_jobs": -1,
    "verbosity": 0
}


num_boost_round = 100

### Evaluation function

In [11]:
def evaluate_model(model, X_val, y_val):
    y_proba = model.predict(
        xgb.DMatrix(X_val)
    )

    y_pred = (
        y_proba >= 0.5
    ).astype(int)

    pr_auc = average_precision_score(
        y_val,
        y_proba
    )

    roc_auc = roc_auc_score(
        y_val,
        y_proba
    )

    report = classification_report(
        y_val,
        y_pred,
        digits=4
    )

    return {
        "y_true": np.asarray(y_val),
        "y_proba": np.asarray(y_proba),
        "y_pred": y_pred,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "classification_report": report
    }

---

### Stratified Group K-Fold

In [12]:
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

sgkf_results = []
sgkf_predictions = []

X = df[features]
y = df["label"]
groups = df["profile_date"]

for fold, (train_idx, val_idx) in enumerate(
    sgkf.split(X, y, groups),
    start=1
):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    train_days = sorted(
        df.iloc[train_idx]["profile_date"].unique()
    )

    validation_days = sorted(
        df.iloc[val_idx]["profile_date"].unique()
    )

    fold_params = params.copy()
    fold_params["scale_pos_weight"] = (
        (y_train == 0).sum()
        /
        (y_train == 1).sum()
    )

    model = xgb.train(
        params=fold_params,
        dtrain=xgb.DMatrix(
            X_train,
            label=y_train
        ),
        num_boost_round=num_boost_round
    )

    results = evaluate_model(
        model,
        X_val,
        y_val
    )

    sgkf_results.append({
        "fold": fold,
        "train_days": train_days,
        "validation_days": validation_days,
        "pr_auc": results["pr_auc"],
        "roc_auc": results["roc_auc"],
        "classification_report": results["classification_report"]
    })

    sgkf_predictions.append({
        "fold": fold,
        "y_true": results["y_true"],
        "y_proba": results["y_proba"],
        "y_pred": results["y_pred"]
    })

    print(f"\n{'=' * 60}")
    print(f"SGKF Fold {fold}")
    print(f"Training days: {train_days}")
    print(f"Validation days: {validation_days}")
    print(f"PR-AUC: {results['pr_auc']:.4f}")
    print(f"ROC-AUC: {results['roc_auc']:.4f}")
    print("\nClassification Report:")
    print(results["classification_report"])

# Fold Summary
sgkf_summary = pd.DataFrame([
    {
        "fold": result["fold"],
        "train_days": result["train_days"],
        "validation_days": result["validation_days"],
        "pr_auc": result["pr_auc"],
        "roc_auc": result["roc_auc"]
    }
    for result in sgkf_results
])

display(sgkf_summary)

print(f"Mean PR-AUC: {sgkf_summary['pr_auc'].mean():.4f}")
print(f"Mean ROC-AUC: {sgkf_summary['roc_auc'].mean():.4f}")


SGKF Fold 1
Training days: [Timestamp('2022-11-01 00:00:00'), Timestamp('2022-11-02 00:00:00'), Timestamp('2022-11-03 00:00:00'), Timestamp('2022-11-04 00:00:00'), Timestamp('2022-11-05 00:00:00'), Timestamp('2022-11-06 00:00:00'), Timestamp('2022-11-08 00:00:00'), Timestamp('2022-11-09 00:00:00'), Timestamp('2022-11-11 00:00:00')]
Validation days: [Timestamp('2022-11-07 00:00:00'), Timestamp('2022-11-10 00:00:00')]
PR-AUC: 0.4360
ROC-AUC: 0.8642

Classification Report:
              precision    recall  f1-score   support

           0     0.9815    0.7670    0.8611     54106
           1     0.1690    0.7662    0.2769      3345

    accuracy                         0.7670     57451
   macro avg     0.5752    0.7666    0.5690     57451
weighted avg     0.9342    0.7670    0.8271     57451


SGKF Fold 2
Training days: [Timestamp('2022-11-01 00:00:00'), Timestamp('2022-11-03 00:00:00'), Timestamp('2022-11-04 00:00:00'), Timestamp('2022-11-05 00:00:00'), Timestamp('2022-11-06 00:00:00')

,fold,train_days,validation_days,pr_auc,roc_auc
0,1,"[2022-11-01 00:00:00, 2022-11-02 00:00:00, 202...","[2022-11-07 00:00:00, 2022-11-10 00:00:00]",0.436014,0.864156
1,2,"[2022-11-01 00:00:00, 2022-11-03 00:00:00, 202...","[2022-11-02 00:00:00, 2022-11-11 00:00:00]",0.448312,0.866567
2,3,"[2022-11-01 00:00:00, 2022-11-02 00:00:00, 202...","[2022-11-04 00:00:00, 2022-11-05 00:00:00]",0.420980,0.861871
3,4,"[2022-11-02 00:00:00, 2022-11-03 00:00:00, 202...","[2022-11-01 00:00:00, 2022-11-06 00:00:00, 202...",0.401602,0.857523
4,5,"[2022-11-01 00:00:00, 2022-11-02 00:00:00, 202...","[2022-11-03 00:00:00, 2022-11-09 00:00:00]",0.431597,0.864355


Mean PR-AUC: 0.4277
Mean ROC-AUC: 0.8629


---

### Rolling Validation

In [13]:
rolling_results = []
rolling_predictions = []

days = sorted(
    df["profile_date"].unique()
)

for i in range(8, len(days)):

    train_days = days[:i]
    validation_day = days[i]

    train_data = df[
        df["profile_date"].isin(train_days)
    ]

    validation_data = df[
        df["profile_date"] == validation_day
    ]

    X_train = train_data[features]
    y_train = train_data["label"]

    X_val = validation_data[features]
    y_val = validation_data["label"]

    fold_params = params.copy()
    fold_params["scale_pos_weight"] = (
        (y_train == 0).sum()
        /
        (y_train == 1).sum()
    )

    model = xgb.train(
        params=fold_params,
        dtrain=xgb.DMatrix(
            X_train,
            label=y_train
        ),
        num_boost_round=num_boost_round
    )

    results = evaluate_model(
        model,
        X_val,
        y_val
    )

    rolling_results.append({
        "training_period": (
            f"{train_days[0]} → {train_days[-1]}"
        ),
        "validation_day": validation_day,
        "pr_auc": results["pr_auc"],
        "roc_auc": results["roc_auc"],
        "classification_report": results["classification_report"]
    })

    rolling_predictions.append({
        "validation_day": validation_day,
        "y_true": results["y_true"],
        "y_proba": results["y_proba"],
        "y_pred": results["y_pred"]
    })

    print(f"\n{'=' * 60}")
    print(f"Rolling validation: {validation_day}")
    print(
        f"Training: {train_days[0]} → {train_days[-1]}"
    )
    print(f"PR-AUC: {results['pr_auc']:.4f}")
    print(f"ROC-AUC: {results['roc_auc']:.4f}")
    print("\nClassification Report:")
    print(results["classification_report"])

# Fold summary
rolling_summary = pd.DataFrame([
    {
        "training_period": result["training_period"],
        "validation_day": result["validation_day"],
        "pr_auc": result["pr_auc"],
        "roc_auc": result["roc_auc"]
    }
    for result in rolling_results
])

display(rolling_summary)

print(f"Mean PR-AUC: {rolling_summary['pr_auc'].mean():.4f}")
print(f"Mean ROC-AUC: {rolling_summary['roc_auc'].mean():.4f}")


Rolling validation: 2022-11-09 00:00:00
Training: 2022-11-01 00:00:00 → 2022-11-08 00:00:00
PR-AUC: 0.4276
ROC-AUC: 0.8618

Classification Report:
              precision    recall  f1-score   support

           0     0.9807    0.7738    0.8651     27070
           1     0.1728    0.7564    0.2813      1691

    accuracy                         0.7728     28761
   macro avg     0.5768    0.7651    0.5732     28761
weighted avg     0.9332    0.7728    0.8308     28761


Rolling validation: 2022-11-10 00:00:00
Training: 2022-11-01 00:00:00 → 2022-11-09 00:00:00
PR-AUC: 0.4234
ROC-AUC: 0.8567

Classification Report:
              precision    recall  f1-score   support

           0     0.9805    0.7600    0.8563     26864
           1     0.1640    0.7570    0.2696      1671

    accuracy                         0.7598     28535
   macro avg     0.5723    0.7585    0.5630     28535
weighted avg     0.9327    0.7598    0.8219     28535


Rolling validation: 2022-11-11 00:00:00
Training:

,training_period,validation_day,pr_auc,roc_auc
0,2022-11-01 00:00:00 → 2022-11-08 00:00:00,2022-11-09,0.427614,0.861813
1,2022-11-01 00:00:00 → 2022-11-09 00:00:00,2022-11-10,0.423421,0.856712
2,2022-11-01 00:00:00 → 2022-11-10 00:00:00,2022-11-11,0.465015,0.873992


Mean PR-AUC: 0.4387
Mean ROC-AUC: 0.8642


---

### Stratified K-fold

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=False
)

skf_results = []
skf_predictions = []

X = df[features]
y = df["label"]

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X, y),
    start=1
):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    train_days = sorted(
        df.iloc[train_idx]["profile_date"].unique()
    )

    validation_days = sorted(
        df.iloc[val_idx]["profile_date"].unique()
    )

    fold_params = params.copy()
    fold_params["scale_pos_weight"] = (
        (y_train == 0).sum()
        /
        (y_train == 1).sum()
    )

    model = xgb.train(
        params=fold_params,
        dtrain=xgb.DMatrix(
            X_train,
            label=y_train
        ),
        num_boost_round=num_boost_round
    )

    results = evaluate_model(
        model,
        X_val,
        y_val
    )

    skf_results.append({
        "fold": fold,
        "train_days": train_days,
        "validation_days": validation_days,
        "pr_auc": results["pr_auc"],
        "roc_auc": results["roc_auc"],
        "classification_report": results["classification_report"]
    })

    skf_predictions.append({
        "fold": fold,
        "y_true": results["y_true"],
        "y_proba": results["y_proba"],
        "y_pred": results["y_pred"]
    })

    print(f"\n{'=' * 60}")
    print(f"SKF Fold {fold}")
    print(f"Training days: {train_days}")
    print(f"Validation days: {validation_days}")
    print(f"PR-AUC: {results['pr_auc']:.4f}")
    print(f"ROC-AUC: {results['roc_auc']:.4f}")
    print("\nClassification Report:")
    print(results["classification_report"])

# Fold summary
skf_summary = pd.DataFrame([
    {
        "fold": result["fold"],
        "train_days": result["train_days"],
        "validation_days": result["validation_days"],
        "pr_auc": result["pr_auc"],
        "roc_auc": result["roc_auc"]
    }
    for result in skf_results
])

display(skf_summary)

print(f"Mean PR-AUC: {skf_summary['pr_auc'].mean():.4f}")
print(f"Mean ROC-AUC: {skf_summary['roc_auc'].mean():.4f}")


SKF Fold 1
Training days: [Timestamp('2022-11-01 00:00:00'), Timestamp('2022-11-02 00:00:00'), Timestamp('2022-11-03 00:00:00'), Timestamp('2022-11-04 00:00:00'), Timestamp('2022-11-05 00:00:00'), Timestamp('2022-11-06 00:00:00'), Timestamp('2022-11-07 00:00:00'), Timestamp('2022-11-08 00:00:00'), Timestamp('2022-11-09 00:00:00'), Timestamp('2022-11-10 00:00:00'), Timestamp('2022-11-11 00:00:00')]
Validation days: [Timestamp('2022-11-01 00:00:00'), Timestamp('2022-11-02 00:00:00'), Timestamp('2022-11-03 00:00:00'), Timestamp('2022-11-04 00:00:00'), Timestamp('2022-11-05 00:00:00'), Timestamp('2022-11-06 00:00:00'), Timestamp('2022-11-07 00:00:00'), Timestamp('2022-11-08 00:00:00'), Timestamp('2022-11-09 00:00:00'), Timestamp('2022-11-10 00:00:00'), Timestamp('2022-11-11 00:00:00')]
PR-AUC: 0.4157
ROC-AUC: 0.8598

Classification Report:
              precision    recall  f1-score   support

           0     0.9802    0.7685    0.8615     59297
           1     0.1645    0.7457    0.269

,fold,train_days,validation_days,pr_auc,roc_auc
0,1,"[2022-11-01 00:00:00, 2022-11-02 00:00:00, 202...","[2022-11-01 00:00:00, 2022-11-02 00:00:00, 202...",0.415748,0.859836
1,2,"[2022-11-01 00:00:00, 2022-11-02 00:00:00, 202...","[2022-11-01 00:00:00, 2022-11-02 00:00:00, 202...",0.428396,0.865167
2,3,"[2022-11-01 00:00:00, 2022-11-02 00:00:00, 202...","[2022-11-01 00:00:00, 2022-11-02 00:00:00, 202...",0.423126,0.863856
3,4,"[2022-11-01 00:00:00, 2022-11-02 00:00:00, 202...","[2022-11-01 00:00:00, 2022-11-02 00:00:00, 202...",0.435414,0.865600
4,5,"[2022-11-01 00:00:00, 2022-11-02 00:00:00, 202...","[2022-11-01 00:00:00, 2022-11-02 00:00:00, 202...",0.418544,0.859324


Mean PR-AUC: 0.4242
Mean ROC-AUC: 0.8628


--- 

### Evaluation and Comparison

In [22]:
comparison_results = []

validation_predictions = {
    "SGKF": sgkf_predictions,
    "Rolling": rolling_predictions,
    "SKF": skf_predictions
}

for strategy, predictions in validation_predictions.items():
    y_true = np.concatenate(
        [p["y_true"] for p in predictions]
    )

    y_proba = np.concatenate(
        [p["y_proba"] for p in predictions]
    )

    y_pred = (
        y_proba >= 0.5
    ).astype(int)

    comparison_results.append({
        "strategy": strategy,
        "pr_auc": average_precision_score(
            y_true,
            y_proba
        ),
        "roc_auc": roc_auc_score(
            y_true,
            y_proba
        ),
        "classification_report": classification_report(
            y_true,
            y_pred,
            digits=4
        )
    })

comparison_df = pd.DataFrame(
    comparison_results
)

display(
    comparison_df[
        [
            "strategy",
            "pr_auc",
            "roc_auc"
        ]
    ]
)

for result in comparison_results:
    print(f"\n{'=' * 70}")
    print(f"{result['strategy']} — Overall Classification Report")
    print(f"{'=' * 70}")
    print(result["classification_report"])

,strategy,pr_auc,roc_auc
0,SGKF,0.425542,0.862338
1,Rolling,0.438903,0.864254
2,SKF,0.424101,0.862724



SGKF — Overall Classification Report
              precision    recall  f1-score   support

           0     0.9810    0.7696    0.8626    296481
           1     0.1672    0.7566    0.2739     18129

    accuracy                         0.7689    314610
   macro avg     0.5741    0.7631    0.5682    314610
weighted avg     0.9341    0.7689    0.8286    314610


Rolling — Overall Classification Report
              precision    recall  f1-score   support

           0     0.9811    0.7667    0.8608     81111
           1     0.1699    0.7640    0.2780      5071

    accuracy                         0.7666     86182
   macro avg     0.5755    0.7653    0.5694     86182
weighted avg     0.9334    0.7666    0.8265     86182


SKF — Overall Classification Report
              precision    recall  f1-score   support

           0     0.9812    0.7690    0.8622    296481
           1     0.1672    0.7586    0.2740     18129

    accuracy                         0.7684    314610
   macro avg